# OpenClash 流量统计核对
来源：machome 上 OpenClashTraffic/traffic.sqlite3；通过 SQLite 只读事务查询。
时区：Asia/Shanghai。GB=10^9 bytes。统计到采样截止时点，首尾日期不完整。
每2秒采集连接增量，可能漏计短连接及连接关闭前尾部字节；仅统计经过 Mihomo 且来源属于 LAN 的流量。域名维度只覆盖可识别域名，多个维度不可相加。源IP未映射主机名；已通过 ipconfig 验证 machome=192.168.1.113。
核对：非负值、来源完整性，以及端口/节点/规则维度与日汇总总量一致。下面保留确定排序的3行样本与汇总结果。重跑需 SSH machome 访问权限，会读取更新后的数据。

In [1]:
import subprocess, json
remote = 'import sqlite3,pathlib,json,datetime\np=pathlib.Path.home()/\'Library/Application Support/OpenClashTraffic/traffic.sqlite3\'\nc=sqlite3.connect(\'file:\'+str(p)+\'?mode=ro\',uri=True); c.row_factory=sqlite3.Row;c.execute(\'BEGIN\')\nqueries={\n\'range\':"select min(first_seen_at) first_seen,max(last_seen_at) last_seen,count(distinct day) days from daily_traffic",\n\'periods\':"select \'累计\' period,category,sum(upload_bytes) up,sum(download_bytes) down from daily_traffic group by category union all select \'本月\',category,sum(upload_bytes),sum(download_bytes) from daily_traffic where day>=\'2026-09-01\' group by category union all select \'昨日\',category,sum(upload_bytes),sum(download_bytes) from daily_traffic where day=\'2026-09-07\' group by category union all select \'今日\',category,sum(upload_bytes),sum(download_bytes) from daily_traffic where day=\'2026-09-08\' group by category",\n\'devices\':"select source_ip,sum(upload_bytes) up,sum(download_bytes) down,sum(case when category=\'PROXY\' then upload_bytes+download_bytes else 0 end) proxy from daily_traffic where day>=\'2026-09-01\' group by source_ip order by up+down desc",\n\'daily\':"select day,sum(upload_bytes) up,sum(download_bytes) down from daily_traffic where day>=\'2026-09-01\' group by day",\n\'domains\':"select dimension_value,category,sum(upload_bytes) up,sum(download_bytes) down from daily_dimensions where day>=\'2026-09-01\' and dimension=\'domain\' group by dimension_value,category order by up+down desc limit 20",\n\'outbound\':"select dimension_value,category,sum(upload_bytes) up,sum(download_bytes) down from daily_dimensions where day>=\'2026-09-01\' and dimension=\'outbound\' group by dimension_value,category",\n\'ports\':"select dimension_value,category,sum(upload_bytes) up,sum(download_bytes) down from daily_dimensions where day>=\'2026-09-01\' and dimension=\'destination_port\' group by dimension_value,category order by up+down desc limit 20",\n\'dimension_totals\':"select dimension,sum(upload_bytes) up,sum(download_bytes) down from daily_dimensions where day>=\'2026-09-01\' group by dimension",\n\'preview\':"select day,source_ip,category,upload_bytes,download_bytes from daily_traffic order by day,source_ip,category limit 3",\n\'quality\':"select count(*) rows,sum(upload_bytes<0 or download_bytes<0) negative_rows,sum(source_ip is null or source_ip=\'\') missing_sources from daily_traffic"\n}\nr={k:[dict(x) for x in c.execute(q)] for k,q in queries.items()};r[\'captured_at\']=datetime.datetime.now().isoformat();c.rollback();print(json.dumps(r,ensure_ascii=False))\n'
result = json.loads(subprocess.run(["ssh", "-o", "BatchMode=yes", "machome", "/Users/ellis/miniconda3/bin/python3", "-"], input=remote, text=True, capture_output=True, check=True).stdout)
print(json.dumps(result, ensure_ascii=False, indent=2))


{
  "range": [
    {
      "first_seen": "2026-07-18T07:53:40+00:00",
      "last_seen": "2026-09-08T00:02:56+00:00",
      "days": 53
    }
  ],
  "periods": [
    {
      "period": "累计",
      "category": "DIRECT",
      "up": 640588379,
      "down": 173028994577
    },
    {
      "period": "累计",
      "category": "PROXY",
      "up": 134431517351,
      "down": 131746461995
    },
    {
      "period": "本月",
      "category": "DIRECT",
      "up": 101256116,
      "down": 38529661737
    },
    {
      "period": "本月",
      "category": "PROXY",
      "up": 11442977355,
      "down": 15557684164
    },
    {
      "period": "昨日",
      "category": "DIRECT",
      "up": 10689599,
      "down": 245040244
    },
    {
      "period": "昨日",
      "category": "PROXY",
      "up": 1167548959,
      "down": 2896974754
    },
    {
      "period": "今日",
      "category": "DIRECT",
      "up": 2690835,
      "down": 1954675247
    },
    {
      "period": "今日",
      "category": "PROXY",
  